<a href="https://colab.research.google.com/github/TegarReskiPratama/Penalaran-Komputer-CBR/blob/main/Penalaran_Komputer_CBR_215_209.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install pdfplumber pandas tqdm openpyxl Pillow==9.5.0

In [4]:
import os
import re
import json
import zipfile
import pdfplumber
import pandas as pd
import numpy as np
import joblib

from tqdm import tqdm
from datetime import datetime

In [5]:
from google.colab import files

uploaded = files.upload()

Saving PUTUSAN PIDANA KHUSUS PK_209 DAN 215-20260620T144127Z-3-001.zip to PUTUSAN PIDANA KHUSUS PK_209 DAN 215-20260620T144127Z-3-001 (1).zip


In [6]:
ZIP_FILE = "/content/PUTUSAN PIDANA KHUSUS PK_209 DAN 215-20260620T144127Z-3-001.zip"

EXTRACT_DIR = "/content/dataset"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("ZIP berhasil diekstrak")

ZIP berhasil diekstrak


In [7]:
pdf_files = []

for root, dirs, files in os.walk(EXTRACT_DIR):

    for file in files:

        if file.lower().endswith(".pdf"):

            pdf_files.append(
                os.path.join(root, file)
            )

print("Jumlah PDF ditemukan :", len(pdf_files))

Jumlah PDF ditemukan : 35


In [8]:
if len(pdf_files) < 30:

    raise Exception(
        "Dataset kurang dari 30 dokumen!"
    )

print("Validasi jumlah dokumen berhasil")

Validasi jumlah dokumen berhasil


In [9]:
def extract_pdf_text(pdf_path):

    full_text = ""

    try:

        with pdfplumber.open(pdf_path) as pdf:

            for page in pdf.pages:

                page_text = page.extract_text()

                if page_text:

                    full_text += page_text + "\n"

    except Exception as e:

        print(f"ERROR : {pdf_path}")
        print(e)

    return full_text

In [10]:
def clean_text(text):

    text = re.sub(
        r"mahkamah agung republik indonesia",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"direktori putusan mahkamah agung republik indonesia",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"putusan\.mahkamahagung\.go\.id",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"halaman\s+\d+",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"disclaimer.*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = text.lower()

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [11]:
def tokenize(text):

    tokens = text.split()

    return tokens

In [12]:
import os

PROJECT_DIR = "/content/CBR_PUTUSAN"
RAW_DIR = os.path.join(PROJECT_DIR, "data/raw")
LOG_DIR = os.path.join(PROJECT_DIR, "logs")
PROCESSED_DIR = os.path.join(PROJECT_DIR, "data/processed")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")
EVAL_DIR = os.path.join(PROJECT_DIR, "data/eval")
RESULT_DIR = os.path.join(PROJECT_DIR, "data/results")

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

log_path = os.path.join(
    LOG_DIR,
    "cleaning.log"
)

log_file = open(
    log_path,
    "w",
    encoding="utf-8"
)


In [13]:
dataset_info = []

for idx, pdf_path in enumerate(
    tqdm(pdf_files)
):

    raw_text = extract_pdf_text(
        pdf_path
    )

    cleaned_text = clean_text(
        raw_text
    )

    tokens = tokenize(
        cleaned_text
    )

    word_count = len(tokens)

    txt_name = f"case_{idx+1:03d}.txt"

    txt_path = os.path.join(
        RAW_DIR,
        txt_name
    )

    with open(
        txt_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(cleaned_text)

    dataset_info.append({

        "case_id": idx+1,

        "file_name":
            os.path.basename(pdf_path),

        "word_count":
            word_count,

        "txt_file":
            txt_name

    })

    log_file.write(
        f"{txt_name} | {word_count} words\n"
    )

print("Konversi selesai")

100%|██████████| 35/35 [01:07<00:00,  1.93s/it]

Konversi selesai


In [14]:
log_file.close()

print("Cleaning log berhasil dibuat")

Cleaning log berhasil dibuat


In [15]:
df_stats = pd.DataFrame(
    dataset_info
)

df_stats.head()

,case_id,file_name,word_count,txt_file
0,1,putusan_1089_k_pid.sus-lh_2026_20260613195640.pdf,8111,case_001.txt
1,2,putusan_5767_k_pid.sus-lh_2025_20260613195650.pdf,5701,case_002.txt
2,3,putusan_1961_pk_pid.sus-lh_2025_20260613200459...,4534,case_003.txt
3,4,putusan_9581_k_pid.sus-lh_2025_20260613200642.pdf,6730,case_004.txt
4,5,putusan_1703_k_pid.sus-lh_2026_20260613200136.pdf,6462,case_005.txt


In [16]:
print(
    "Jumlah Dokumen :",
    len(df_stats)
)

print(
    "Rata-rata Kata :",
    int(df_stats["word_count"].mean())
)

print(
    "Minimum Kata :",
    int(df_stats["word_count"].min())
)

print(
    "Maximum Kata :",
    int(df_stats["word_count"].max())
)

Jumlah Dokumen : 35
Rata-rata Kata : 5996
Minimum Kata : 3278
Maximum Kata : 16756


In [17]:
stats_path = os.path.join(
    PROJECT_DIR,
    "data",
    "dataset_statistics.csv"
)

df_stats.to_csv(
    stats_path,
    index=False
)

print(
    "dataset_statistics.csv berhasil dibuat"
)

dataset_statistics.csv berhasil dibuat


In [18]:
MIN_WORDS = 500

valid_docs = 0

for wc in df_stats["word_count"]:

    if wc >= MIN_WORDS:

        valid_docs += 1

percentage = (
    valid_docs /
    len(df_stats)
) * 100

print(
    f"Dokumen valid : {percentage:.2f}%"
)

Dokumen valid : 100.00%


In [19]:
print("="*50)

print("CASE BASE BERHASIL DIBANGUN")

print("="*50)

print("Jumlah PDF :", len(pdf_files))

print("TXT :", len(os.listdir(RAW_DIR)))

print("Cleaning Log :", log_path)

print("="*50)

CASE BASE BERHASIL DIBANGUN
Jumlah PDF : 35
TXT : 35
Cleaning Log : /content/CBR_PUTUSAN/logs/cleaning.log


# NOTEBOOK 2

In [20]:
def extract_nomor_perkara(text):

    pattern = r"\d+\s*[A-Z]*\/Pid\.Sus\-LH\/\d{4}"

    match = re.search(
        pattern,
        text,
        re.IGNORECASE
    )

    if match:
        return match.group()

    return ""

In [21]:
def extract_tahun(text):

    nomor = extract_nomor_perkara(text)

    tahun = re.findall(
        r"\d{4}",
        nomor
    )

    if tahun:
        return tahun[-1]

    return ""

In [22]:
def extract_pasal(text):

    pasal_list = re.findall(
        r"pasal\s+\d+",
        text,
        re.IGNORECASE
    )

    pasal_list = list(
        set(pasal_list)
    )

    return "; ".join(
        pasal_list
    )

In [23]:
def extract_terdakwa(text):

    patterns = [

        r"terdakwa\s*[:\-]\s*(.*?)\n",

        r"nama\s*[:\-]\s*(.*?)\n"

    ]

    for p in patterns:

        match = re.search(
            p,
            text,
            re.IGNORECASE
        )

        if match:

            return match.group(1)

    return ""

In [24]:
def extract_jenis_perkara(text):

    if "pid.sus-lh" in text:

        return "Pidana Khusus Lingkungan Hidup"

    return "Pidana Khusus"

In [25]:
def extract_tanggal(text):

    pattern = r"\d{1,2}\s+[a-zA-Z]+\s+\d{4}"

    match = re.search(
        pattern,
        text
    )

    if match:
        return match.group()

    return ""

In [26]:
def extract_ringkasan_fakta(text):

    words = text.split()

    return " ".join(
        words[:1000]
    )

In [27]:
def extract_amar_putusan(text):

    patterns = [

        r"mengadili(.*?)(?:demikian|$)",

        r"amar putusan(.*?)(?:demikian|$)"

    ]

    for p in patterns:

        match = re.search(
            p,
            text,
            re.IGNORECASE |
            re.DOTALL
        )

        if match:

            return match.group(1)[:2000]

    return ""

In [28]:
def extract_argumen_hukum(text):

    words = text.split()

    start = int(
        len(words)*0.3
    )

    end = int(
        len(words)*0.6
    )

    return " ".join(
        words[start:end]
    )

In [29]:
def get_word_count(text):

    return len(
        text.split()
    )

In [30]:
def get_char_count(text):

    return len(text)

In [31]:
def unique_words(text):

    return len(
        set(text.split())
    )

# Build Dataset

In [32]:
records = []

txt_files = [f for f in os.listdir(RAW_DIR) if f.endswith('.txt')]

for idx, txt_file in enumerate(
    tqdm(txt_files)
):

    path = os.path.join(
        RAW_DIR,
        txt_file
    )

    with open(
        path,
        encoding="utf-8"
    ) as f:

        text = f.read()

    record = {

        "case_id":
            idx+1,

        "file_name":
            txt_file,

        "nomor_perkara":
            extract_nomor_perkara(text),

        "tahun":
            extract_tahun(text),

        "tanggal":
            extract_tanggal(text),

        "jenis_perkara":
            extract_jenis_perkara(text),

        "terdakwa":
            extract_terdakwa(text),

        "pasal":
            extract_pasal(text),

        "ringkasan_fakta":
            extract_ringkasan_fakta(text),

        "argumen_hukum":
            extract_argumen_hukum(text),

        "amar_putusan":
            extract_amar_putusan(text),

        "word_count":
            get_word_count(text),

        "char_count":
            get_char_count(text),

        "unique_words":
            unique_words(text),

        "text_full":
            text
    }

    records.append(
        record
    )


100%|██████████| 35/35 [00:00<00:00, 149.39it/s]


In [33]:
cases_df = pd.DataFrame(
    records
)

cases_df.head()

,case_id,file_name,nomor_perkara,tahun,tanggal,jenis_perkara,terdakwa,pasal,ringkasan_fakta,argumen_hukum,amar_putusan,word_count,char_count,unique_words,text_full
0,1,case_021.txt,,,1 januari 1970,Pidana Khusus,,pasal 78; pasal 30; pasal 167; pasal 55; pasal...,a i s e n o d n i k i l b u p e a r i s g e n ...,r i s g e n n o u d g n a i h k a i l m b u a ...,perkara terdakwa; i l − bahwa putusan judex f...,5539,27855,827,a i s e n o d n i k i l b u p e a r i s g e n ...
1,2,case_031.txt,2531 k/pid.sus-lh/2024,2024,13 april 1971,Pidana Khusus Lingkungan Hidup,,pasal 253; pasal 197; pasal 158; pasal 35; pas...,a i s e n o d n i k i l b u p e a r i s g e n ...,e dari 8 halaman putusan nomor 2531 k/pid.sus-...,", sehingga amar selengkapnya berbunyi seibagai...",4700,22926,667,a i s e n o d n i k i l b u p e a r i s g e n ...
2,3,case_028.txt,2630 pk/pid.sus-lh/2025,2025,10 agustusi 1984,Pidana Khusus Lingkungan Hidup,,pasal 105; pasal 55; pasal 161; pasal 197; pas...,a i s e n o d n i k i l b u p e a r i s g e n ...,a i h k a i l m b u a p k direktori putusan e ...,kembali a perkara tersebut dengan amar sepert...,5586,27753,824,a i s e n o d n i k i l b u p e a r i s g e n ...
3,4,case_017.txt,205 pk/pid.sus-lh/2026,2026,12 april 1977,Pidana Khusus Lingkungan Hidup,,pasal 111; pasal 15; pasal 322; pasal 159; pas...,a i s e n o d n i k i l b u p e a r i s g e n ...,negeri makassar nomor 474/pid. e sus/2023/pn m...,sendiri: i h 1. menyatakan terdakwa helmut he...,6122,31206,946,a i s e n o d n i k i l b u p e a r i s g e n ...
4,5,case_025.txt,2071 k/pid.sus-lh/2026,2026,28 februari 1961,Pidana Khusus Lingkungan Hidup,,pasal 253; pasal 8; pasal 35; pasal 345; pasal...,a i s e n o d n i k i l b u p e a r i s g e n ...,k/pid.sus-lh/2026 e n n o u d g n a i h k kepa...,t erdakwa sesuai hukum acara pidana yang berl...,8779,44853,1191,a i s e n o d n i k i l b u p e a r i s g e n ...


In [34]:
print(
    "Jumlah Kasus:",
    len(cases_df)
)

print(
    "Rata-rata Kata:",
    int(
        cases_df["word_count"]
        .mean()
    )
)

Jumlah Kasus: 35
Rata-rata Kata: 5996


In [35]:
csv_path = os.path.join(
    PROCESSED_DIR,
    "cases.csv"
)

cases_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "cases.csv berhasil dibuat"
)

cases.csv berhasil dibuat


In [36]:
json_path = os.path.join(
    PROCESSED_DIR,
    "cases.json"
)

cases_df.to_json(
    json_path,
    orient="records",
    force_ascii=False,
    indent=4
)

print(
    "cases.json berhasil dibuat"
)

cases.json berhasil dibuat


In [37]:
xlsx_path = os.path.join(
    PROCESSED_DIR,
    "metadata.xlsx"
)

cases_df.to_excel(
    xlsx_path,
    index=False
)

print(
    "metadata.xlsx berhasil dibuat"
)

metadata.xlsx berhasil dibuat


In [38]:
cases_df[
    [
        "case_id",
        "nomor_perkara",
        "terdakwa",
        "pasal",
        "tahun"
    ]
].head(10)

,case_id,nomor_perkara,terdakwa,pasal,tahun
0,1,,,pasal 78; pasal 30; pasal 167; pasal 55; pasal...,
1,2,2531 k/pid.sus-lh/2024,,pasal 253; pasal 197; pasal 158; pasal 35; pas...,2024
2,3,2630 pk/pid.sus-lh/2025,,pasal 105; pasal 55; pasal 161; pasal 197; pas...,2025
3,4,205 pk/pid.sus-lh/2026,,pasal 111; pasal 15; pasal 322; pasal 159; pas...,2026
4,5,2071 k/pid.sus-lh/2026,,pasal 253; pasal 8; pasal 35; pasal 345; pasal...,2026
5,6,4305 k/pid.sus-lh/2026,,pasal 253; pasal 55; pasal 3; pasal 158; pasal...,2026
6,7,,,pasal 253; pasal 136; pasal 55; pasal 197; pas...,
7,8,5851k/pid.sus-lh/2024,,pasal 55; pasal 35; pasal 14; pasal 158,2024
8,9,1703 k/pid.sus-lh/2026,,pasal 3; pasal 197; pasal 361; pasal 158; pasa...,2026
9,10,496 pk/pid.sus-lh/2024,,pasal 56; pasal 55; pasal 266; pasal 158; pasa...,2024


In [39]:
print("="*50)

print(
    "CASE REPRESENTATION SELESAI"
)

print("="*50)

print(
    "CSV :",
    csv_path
)

print(
    "JSON:",
    json_path
)

print(
    "XLSX:",
    xlsx_path
)

print("="*50)

CASE REPRESENTATION SELESAI
CSV : /content/CBR_PUTUSAN/data/processed/cases.csv
JSON: /content/CBR_PUTUSAN/data/processed/cases.json
XLSX: /content/CBR_PUTUSAN/data/processed/metadata.xlsx


# Notebook 3

In [40]:
!pip install sentence-transformers transformers torch joblib

In [41]:
from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics.pairwise import cosine_similarity

from sklearn.svm import LinearSVC

from sklearn.naive_bayes import MultinomialNB

from sentence_transformers import SentenceTransformer

from tqdm import tqdm

In [42]:
PROJECT_DIR = "/content/CBR_PUTUSAN"

PROCESSED_DIR = os.path.join(
    PROJECT_DIR,
    "data/processed"
)

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "models"
)

EVAL_DIR = os.path.join(
    PROJECT_DIR,
    "data/eval"
)

RESULT_DIR = os.path.join(
    PROJECT_DIR,
    "data/results"
)

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

In [43]:
cases_df = pd.read_csv(
    os.path.join(
        PROCESSED_DIR,
        "cases.csv"
    )
)

print(cases_df.shape)

cases_df.head()

(35, 15)


,case_id,file_name,nomor_perkara,tahun,tanggal,jenis_perkara,terdakwa,pasal,ringkasan_fakta,argumen_hukum,amar_putusan,word_count,char_count,unique_words,text_full
0,1,case_021.txt,NaN,NaN,1 januari 1970,Pidana Khusus,NaN,pasal 78; pasal 30; pasal 167; pasal 55; pasal...,a i s e n o d n i k i l b u p e a r i s g e n ...,r i s g e n n o u d g n a i h k a i l m b u a ...,perkara terdakwa; i l − bahwa putusan judex f...,5539,27855,827,a i s e n o d n i k i l b u p e a r i s g e n ...
1,2,case_031.txt,2531 k/pid.sus-lh/2024,2024.0,13 april 1971,Pidana Khusus Lingkungan Hidup,NaN,pasal 253; pasal 197; pasal 158; pasal 35; pas...,a i s e n o d n i k i l b u p e a r i s g e n ...,e dari 8 halaman putusan nomor 2531 k/pid.sus-...,", sehingga amar selengkapnya berbunyi seibagai...",4700,22926,667,a i s e n o d n i k i l b u p e a r i s g e n ...
2,3,case_028.txt,2630 pk/pid.sus-lh/2025,2025.0,10 agustusi 1984,Pidana Khusus Lingkungan Hidup,NaN,pasal 105; pasal 55; pasal 161; pasal 197; pas...,a i s e n o d n i k i l b u p e a r i s g e n ...,a i h k a i l m b u a p k direktori putusan e ...,kembali a perkara tersebut dengan amar sepert...,5586,27753,824,a i s e n o d n i k i l b u p e a r i s g e n ...
3,4,case_017.txt,205 pk/pid.sus-lh/2026,2026.0,12 april 1977,Pidana Khusus Lingkungan Hidup,NaN,pasal 111; pasal 15; pasal 322; pasal 159; pas...,a i s e n o d n i k i l b u p e a r i s g e n ...,negeri makassar nomor 474/pid. e sus/2023/pn m...,sendiri: i h 1. menyatakan terdakwa helmut he...,6122,31206,946,a i s e n o d n i k i l b u p e a r i s g e n ...
4,5,case_025.txt,2071 k/pid.sus-lh/2026,2026.0,28 februari 1961,Pidana Khusus Lingkungan Hidup,NaN,pasal 253; pasal 8; pasal 35; pasal 345; pasal...,a i s e n o d n i k i l b u p e a r i s g e n ...,k/pid.sus-lh/2026 e n n o u d g n a i h k kepa...,t erdakwa sesuai hukum acara pidana yang berl...,8779,44853,1191,a i s e n o d n i k i l b u p e a r i s g e n ...


In [44]:
cases_df["document"] = (

    cases_df["ringkasan_fakta"].fillna("") +

    " " +

    cases_df["argumen_hukum"].fillna("") +

    " " +

    cases_df["amar_putusan"].fillna("") +

    " " +

    cases_df["pasal"].fillna("")

)

### Labeling Data


In [45]:
def create_label(text):

    text = str(text).lower()

    if "menolak" in text:
        return "DITOLAK"

    elif "mengabulkan" in text:
        return "DIKABULKAN"

    elif "memperbaiki" in text:
        return "DIPERBAIKI"

    elif "membatalkan" in text:
        return "DIBATALKAN"

    elif "lepas" in text:
        return "LEPAS"

    elif "bebas" in text:
        return "BEBAS"

    elif "menguatkan" in text:
        return "DIKUATKAN"

    else:
        return "LAINNYA"


cases_df["label"] = cases_df[
    "amar_putusan"
].apply(create_label)

print(cases_df["label"].value_counts())

label
LAINNYA       28
DITOLAK        4
DIKUATKAN      1
DIKABULKAN     1
DIPERBAIKI     1
Name: count, dtype: int64


In [46]:
print("="*50)

print("Distribusi Label")

print("="*50)

print(
    cases_df["label"].value_counts()
)

print("="*50)

print(
    "Jumlah kelas:",
    cases_df["label"].nunique()
)

Distribusi Label
label
LAINNYA       28
DITOLAK        4
DIKUATKAN      1
DIKABULKAN     1
DIPERBAIKI     1
Name: count, dtype: int64
Jumlah kelas: 5


In [47]:
X_train, X_test, y_train, y_test = train_test_split(

    cases_df["document"],

    cases_df["label"],

    test_size=0.20,

    random_state=42

)

print(
    "Train:",
    len(X_train)
)

print(
    "Test:",
    len(X_test)
)

Train: 28
Test: 7


## TF-IDF

In [48]:
tfidf = TfidfVectorizer(

    max_features=10000,

    ngram_range=(1,2),

    min_df=2

)

In [49]:
X_train_tfidf = tfidf.fit_transform(
    X_train
)

X_test_tfidf = tfidf.transform(
    X_test
)

print(X_train_tfidf.shape)

(28, 3493)


In [50]:
joblib.dump(

    tfidf,

    os.path.join(
        MODEL_DIR,
        "tfidf_vectorizer.pkl"
    )

)

['/content/CBR_PUTUSAN/models/tfidf_vectorizer.pkl']

## SVM

In [51]:
svm_model = LinearSVC(
    random_state=42
)

svm_model.fit(
    X_train_tfidf,
    y_train
)

LinearSVC(random_state=42)

In [52]:
joblib.dump(

    svm_model,

    os.path.join(
        MODEL_DIR,
        "svm_model.pkl"
    )

)

['/content/CBR_PUTUSAN/models/svm_model.pkl']

## NAIVE BAYES

In [53]:
nb_model = MultinomialNB()

nb_model.fit(
    X_train_tfidf,
    y_train
)

MultinomialNB()

In [54]:
joblib.dump(

    nb_model,

    os.path.join(
        MODEL_DIR,
        "nb_model.pkl"
    )

)

['/content/CBR_PUTUSAN/models/nb_model.pkl']

## TF-IDF RETRIEVAL

In [55]:
tfidf_full = tfidf.fit_transform(
    cases_df["document"]
)

In [56]:
def retrieve_tfidf(
        query,
        k=5
):

    q_vec = tfidf.transform(
        [query]
    )

    similarities = cosine_similarity(
        q_vec,
        tfidf_full
    )[0]

    top_idx = similarities.argsort()[::-1][:k]

    results = []

    for idx in top_idx:

        results.append({

            "case_id":
                int(
                    cases_df.iloc[idx]["case_id"]
                ),

            "score":
                float(
                    similarities[idx]
                ),

            "pasal":
                cases_df.iloc[idx]["pasal"]

        })

    return results

In [57]:
query = """
penambangan tanpa izin
menggunakan alat berat
"""

retrieve_tfidf(query)

[{'case_id': 22, 'score': 0.18197953165204642, 'pasal': 'pasal 35; pasal 158'},
 {'case_id': 21,
  'score': 0.13986142568868923,
  'pasal': 'pasal 253; pasal 8; pasal 197; pasal 161; pasal 158'},
 {'case_id': 11,
  'score': 0.1326369791286908,
  'pasal': 'pasal 197; pasal 39; pasal 158; pasal 35; pasal 46'},
 {'case_id': 32,
  'score': 0.09794789860853137,
  'pasal': 'pasal 253; pasal 8; pasal 197; pasal 2; pasal 158; pasal 35'},
 {'case_id': 2,
  'score': 0.09523089908383363,
  'pasal': 'pasal 253; pasal 197; pasal 158; pasal 35; pasal 160'}]

# INDOBERT EMBEDDING

In [58]:
bert_model = SentenceTransformer(

    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [59]:
documents = cases_df[
    "document"
].tolist()

embeddings = bert_model.encode(

    documents,

    show_progress_bar=True,

    convert_to_numpy=True

)

print(embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(35, 384)


In [60]:
np.save(

    os.path.join(
        MODEL_DIR,
        "bert_embeddings.npy"
    ),

    embeddings

)

# BERT RETRIEVAL

In [61]:
def retrieve_bert(
        query,
        k=5
):

    query_embedding = bert_model.encode(
        [query]
    )

    similarities = cosine_similarity(

        query_embedding,

        embeddings

    )[0]

    top_idx = similarities.argsort()[::-1][:k]

    results = []

    for idx in top_idx:

        results.append({

            "case_id":
                int(
                    cases_df.iloc[idx]["case_id"]
                ),

            "score":
                float(
                    similarities[idx]
                ),

            "pasal":
                cases_df.iloc[idx]["pasal"]

        })

    return results

In [62]:
retrieve_bert(
    "penambangan ilegal dalam kawasan hutan"
)

[{'case_id': 35,
  'score': 0.33899882435798645,
  'pasal': 'pasal 253; pasal 8; pasal 3; pasal 197; pasal 361; pasal 618; pasal 162'},
 {'case_id': 9,
  'score': 0.3363853693008423,
  'pasal': 'pasal 3; pasal 197; pasal 361; pasal 158; pasal 618'},
 {'case_id': 5,
  'score': 0.33622482419013977,
  'pasal': 'pasal 253; pasal 8; pasal 35; pasal 345; pasal 55; pasal 3; pasal 197; pasal 81; pasal 361; pasal 158; pasal 618; pasal 20'},
 {'case_id': 21,
  'score': 0.3307664394378662,
  'pasal': 'pasal 253; pasal 8; pasal 197; pasal 161; pasal 158'},
 {'case_id': 23,
  'score': 0.3257811665534973,
  'pasal': 'pasal 55; pasal 263; pasal 266; pasal 158'}]

In [63]:
queries = [

    {
        "query_id":1,
        "query":
        "penambangan tanpa izin"
    },

    {
        "query_id":2,
        "query":
        "pengangkutan mineral ilegal"
    },

    {
        "query_id":3,
        "query":
        "kegiatan tambang dalam kawasan hutan"
    },

    {
        "query_id":4,
        "query":
        "laporan produksi tidak benar"
    },

    {
        "query_id":5,
        "query":
        "eksploitasi nikel ilegal"
    }

]

In [64]:
with open(

    os.path.join(
        EVAL_DIR,
        "queries.json"
    ),

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        queries,

        f,

        indent=4,

        ensure_ascii=False

    )

print("queries.json berhasil dibuat")

queries.json berhasil dibuat


In [65]:
tfidf_results = []

for q in queries:

    retrieved = retrieve_tfidf(
        q["query"]
    )

    tfidf_results.append({

        "query_id":
            q["query_id"],

        "query":
            q["query"],

        "top_case":
            retrieved[0]["case_id"]

    })

In [66]:
pd.DataFrame(
    tfidf_results
).to_csv(

    os.path.join(
        RESULT_DIR,
        "tfidf_topk.csv"
    ),

    index=False

)

In [67]:
bert_results = []

for q in queries:

    retrieved = retrieve_bert(
        q["query"]
    )

    bert_results.append({

        "query_id":
            q["query_id"],

        "query":
            q["query"],

        "top_case":
            retrieved[0]["case_id"]

    })

In [68]:
pd.DataFrame(
    bert_results
).to_csv(

    os.path.join(
        RESULT_DIR,
        "bert_topk.csv"
    ),

    index=False

)

In [69]:
print("="*60)

print("RETRIEVAL BERHASIL")

print("="*60)

print("TF-IDF")
print("SVM")
print("Naive Bayes")
print("IndoBERT")

print("="*60)

RETRIEVAL BERHASIL
TF-IDF
SVM
Naive Bayes
IndoBERT


# Notebook 04 — Retrieval IndoBERT

In [70]:
PROJECT_DIR = "/content/CBR_PUTUSAN"

PROCESSED_DIR = os.path.join(
    PROJECT_DIR,
    "data/processed"
)

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "models"
)

RESULT_DIR = os.path.join(
    PROJECT_DIR,
    "data/results"
)

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)

In [71]:
cases_df = pd.read_csv(

    os.path.join(
        PROCESSED_DIR,
        "cases.csv"
    )

)

print(cases_df.shape)

(35, 15)


In [72]:
cases_df["document"] = (

    cases_df["ringkasan_fakta"]
    .fillna("")

    + " " +

    cases_df["argumen_hukum"]
    .fillna("")

    + " " +

    cases_df["amar_putusan"]
    .fillna("")

    + " " +

    cases_df["pasal"]
    .fillna("")

)

In [73]:
bert_model = SentenceTransformer(

    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [74]:
bert_model = SentenceTransformer(
    "indobenchmark/indobert-base-p1"
)

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

# Generate Embedding

In [75]:
documents = cases_df[
    "document"
].tolist()

embeddings = bert_model.encode(

    documents,

    show_progress_bar=True,

    convert_to_numpy=True

)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [76]:
print(
    embeddings.shape
)

(35, 768)


In [77]:
embedding_path = os.path.join(

    MODEL_DIR,

    "bert_embeddings.npy"

)

np.save(
    embedding_path,
    embeddings
)

print(
    "Embedding berhasil disimpan"
)

Embedding berhasil disimpan


In [78]:
embeddings = np.load(

    os.path.join(
        MODEL_DIR,
        "bert_embeddings.npy"
    )

)

In [79]:
def retrieve_bert(
        query,
        k=5
):

    query_embedding = bert_model.encode(
        [query]
    )

    similarity_scores = cosine_similarity(

        query_embedding,

        embeddings

    )[0]

    top_indices = similarity_scores.argsort()[::-1][:k]

    results = []

    for idx in top_indices:

        results.append({

            "case_id":

                int(
                    cases_df.iloc[idx]["case_id"]
                ),

            "nomor_perkara":

                cases_df.iloc[idx][
                    "nomor_perkara"
                ],

            "pasal":

                cases_df.iloc[idx][
                    "pasal"
                ],

            "score":

                float(
                    similarity_scores[idx]
                )

        })

    return pd.DataFrame(
        results
    )

In [80]:
query = """

penambangan tanpa izin
menggunakan alat berat
dan mengangkut hasil tambang

"""

retrieve_bert(
    query
)

,case_id,nomor_perkara,pasal,score
0,26,482 pk/pid.sus-lh/2024,pasal 263; pasal 35; pasal 266; pasal 158,0.279503
1,22,6191 k/pid.sus-lh/2023,pasal 35; pasal 158,0.277585
2,14,9581 k/pid.sus-lh/2025,pasal 55; pasal 197; pasal 253; pasal 158,0.274415
3,24,1340 k/pid.sus-lh/2026,pasal 253; pasal 3; pasal 197; pasal 359; pasa...,0.270688
4,23,458 pk/pid.sus-lh/2025,pasal 55; pasal 263; pasal 266; pasal 158,0.270409


In [81]:
retrieve_bert(
    query,
    k=10
)

,case_id,nomor_perkara,pasal,score
0,26,482 pk/pid.sus-lh/2024,pasal 263; pasal 35; pasal 266; pasal 158,0.279503
1,22,6191 k/pid.sus-lh/2023,pasal 35; pasal 158,0.277585
2,14,9581 k/pid.sus-lh/2025,pasal 55; pasal 197; pasal 253; pasal 158,0.274415
3,24,1340 k/pid.sus-lh/2026,pasal 253; pasal 3; pasal 197; pasal 359; pasa...,0.270688
4,23,458 pk/pid.sus-lh/2025,pasal 55; pasal 263; pasal 266; pasal 158,0.270409
5,7,NaN,pasal 253; pasal 136; pasal 55; pasal 197; pas...,0.268829
6,9,1703 k/pid.sus-lh/2026,pasal 3; pasal 197; pasal 361; pasal 158; pasa...,0.266825
7,34,6678 k/pid.sus-lh/2024,pasal 35; pasal 8; pasal 158,0.264993
8,13,NaN,pasal 35; pasal 253; pasal 254; pasal 55; pasa...,0.261803
9,3,2630 pk/pid.sus-lh/2025,pasal 105; pasal 55; pasal 161; pasal 197; pas...,0.261375


In [82]:
result_df = retrieve_bert(

    query,

    k=10

)

result_df.to_csv(

    os.path.join(

        RESULT_DIR,

        "bert_retrieval_results.csv"

    ),

    index=False

)

## Batch Retrieval

In [83]:
queries = [

    "penambangan tanpa izin",

    "laporan produksi tidak benar",

    "pengangkutan mineral ilegal",

    "eksploitasi nikel ilegal",

    "pertambangan dalam kawasan hutan"

]

In [84]:
all_results = []

for q in queries:

    retrieval = retrieve_bert(
        q
    )

    best_case = retrieval.iloc[0]

    all_results.append({

        "query":
            q,

        "case_id":
            best_case["case_id"],

        "score":
            best_case["score"]

    })

In [85]:
bert_eval_df = pd.DataFrame(
    all_results
)

bert_eval_df

,query,case_id,score
0,penambangan tanpa izin,23,0.289679
1,laporan produksi tidak benar,22,0.289571
2,pengangkutan mineral ilegal,23,0.299715
3,eksploitasi nikel ilegal,23,0.269612
4,pertambangan dalam kawasan hutan,23,0.297444


## Simpan Evaluasi

In [86]:
bert_eval_df.to_csv(

    os.path.join(

        RESULT_DIR,

        "bert_topk.csv"

    ),

    index=False

)

In [87]:
print("="*50)

print(
    "INDOBERT RETRIEVAL BERHASIL"
)

print("="*50)

print(
    "Jumlah Kasus:",
    len(cases_df)
)

print(
    "Embedding:",
    embeddings.shape
)

print("="*50)

INDOBERT RETRIEVAL BERHASIL
Jumlah Kasus: 35
Embedding: (35, 768)


In [88]:
tfidf = joblib.load(

    os.path.join(
        MODEL_DIR,
        "tfidf_vectorizer.pkl"
    )

)

In [89]:
cases_df["document"] = (

    cases_df["ringkasan_fakta"]
    .fillna("")

    + " " +

    cases_df["argumen_hukum"]
    .fillna("")

    + " " +

    cases_df["amar_putusan"]
    .fillna("")

    + " " +

    cases_df["pasal"]
    .fillna("")

)

In [90]:
tfidf_matrix = tfidf.transform(

    cases_df["document"]

)

In [91]:
bert_model = SentenceTransformer(

    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [92]:
embeddings = np.load(

    os.path.join(
        MODEL_DIR,
        "bert_embeddings.npy"
    )

)

In [131]:
case_solutions = {}
for _, row in cases_df.iterrows():
    case_solutions[
        row["case_id"]
    ] = row["label"] # Use the 'label' column which handles NaN values

# BAGIAN A
# RETRIEVE TF-IDF

In [94]:
def retrieve_tfidf(
        query,
        k=5
):

    q_vec = tfidf.transform(
        [query]
    )

    sim = cosine_similarity(

        q_vec,

        tfidf_matrix

    )[0]

    top_idx = sim.argsort()[::-1][:k]

    results = []

    for idx in top_idx:

        results.append({

            "case_id":

                int(
                    cases_df.iloc[idx]["case_id"]
                ),

            "score":

                float(
                    sim[idx]
                )

        })

    return results

# BAGIAN B
# RETRIEVE BERT

In [95]:
def retrieve_bert(
        query,
        k=5
):

    query_embedding = bert_model.encode(
        [query]
    )

    sim = cosine_similarity(

        query_embedding,

        embeddings

    )[0]

    top_idx = sim.argsort()[::-1][:k]

    results = []

    for idx in top_idx:

        results.append({

            "case_id":

                int(
                    cases_df.iloc[idx]["case_id"]
                ),

            "score":

                float(
                    sim[idx]
                )

        })

    return results

# BAGIAN C
# MAJORITY VOTE

In [96]:
def majority_vote(
        retrieved_cases
):

    solutions = []

    for item in retrieved_cases:

        cid = item["case_id"]

        solusi = case_solutions[cid]

        solutions.append(
            solusi
        )

    return Counter(
        solutions
    ).most_common(1)[0][0]

# BAGIAN D
# WEIGHTED SIMILARITY

In [97]:
def weighted_similarity(
        retrieved_cases
):

    score_dict = {}

    for item in retrieved_cases:

        cid = item["case_id"]

        score = item["score"]

        solution = case_solutions[cid]

        if solution not in score_dict:

            score_dict[
                solution
            ] = 0

        score_dict[
            solution
        ] += score

    best_solution = max(

        score_dict,

        key=score_dict.get

    )

    return best_solution

# BAGIAN E
# PREDICT OUTCOME

In [98]:
def predict_outcome(

        query,

        retrieval_method="bert",

        voting_method="weighted",

        k=5

):

    if retrieval_method == "tfidf":

        retrieved = retrieve_tfidf(
            query,
            k
        )

    else:

        retrieved = retrieve_bert(
            query,
            k
        )

    if voting_method == "majority":

        prediction = majority_vote(
            retrieved
        )

    else:

        prediction = weighted_similarity(
            retrieved
        )

    return prediction

In [99]:
bert_model = SentenceTransformer(
    "indobenchmark/indobert-base-p1"
)

query = """

terdakwa melakukan
penambangan emas ilegal
tanpa izin usaha pertambangan

"""

hasil = predict_outcome(

    query,

    retrieval_method="bert",

    voting_method="weighted"

)

print(hasil)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

nan


# BAGIAN F
# DEMO 5 KASUS BARU

In [100]:
new_cases = [

    "penambangan emas ilegal",

    "penambangan nikel tanpa izin",

    "kegiatan tambang di kawasan hutan",

    "pengangkutan mineral ilegal",

    "laporan produksi palsu"

]

In [101]:
predictions = []

for idx, query in enumerate(
    new_cases
):

    top_cases = retrieve_bert(
        query,
        k=5
    )

    pred = predict_outcome(
        query
    )

    case_ids = [

        x["case_id"]

        for x in top_cases

    ]

    predictions.append({

        "query_id":
            idx+1,

        "query":
            query,

        "predicted_solution":
            pred,

        "top_5_case_ids":
            str(case_ids)

    })

In [102]:
pred_df = pd.DataFrame(
    predictions
)

pred_df

,query_id,query,predicted_solution,top_5_case_ids
0,1,penambangan emas ilegal,NaN,"[23, 14, 7, 22, 13]"
1,2,penambangan nikel tanpa izin,NaN,"[23, 7, 26, 22, 14]"
2,3,kegiatan tambang di kawasan hutan,NaN,"[23, 17, 22, 14, 7]"
3,4,pengangkutan mineral ilegal,NaN,"[23, 14, 7, 22, 13]"
4,5,laporan produksi palsu,NaN,"[17, 22, 23, 14, 13]"


In [103]:
pred_df.to_csv(

    os.path.join(

        RESULT_DIR,

        "predictions.csv"

    ),

    index=False,

    encoding="utf-8-sig"

)

In [104]:
print("="*60)

print(
    "SOLUTION REUSE BERHASIL"
)

print("="*60)

print(
    "Jumlah Query:",
    len(pred_df)
)

print(
    "Predictions:",
    len(pred_df)
)

print("="*60)

SOLUTION REUSE BERHASIL
Jumlah Query: 5
Predictions: 5


# Notebook 06 — Evaluation

## SPLIT DATA

In [105]:
def get_label_from_amar_putusan(text):
    if isinstance(text, str):
        text_lower = text.lower()
        if "ditolak" in text_lower:
            return "DITOLAK"
        elif "dikabulkan" in text_lower:
            return "DIKABULKAN"
        elif "diperbaiki" in text_lower:
            return "DIPERBAIKI"
    return "LAINNYA"

cases_df["label"] = cases_df["amar_putusan"].apply(get_label_from_amar_putusan)

# Ensure all NaN values in 'label' are 'LAINNYA' (if any remain from previous steps)
cases_df["label"] = cases_df["label"].fillna("LAINNYA")

print("Label column created successfully before train_test_split.")
print("\nValue counts for the new 'label' column:")
print(cases_df["label"].value_counts())

from sklearn.model_selection import train_test_split

X = cases_df["document"]

y = cases_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Train :", len(X_train))
print("Test  :", len(X_test))

Label column created successfully before train_test_split.

Value counts for the new 'label' column:
label
LAINNYA    32
DITOLAK     3
Name: count, dtype: int64
Train : 28
Test  : 7


from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=1
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Create tfidf_full with the same vocabulary as the current tfidf
tfidf_full = tfidf.transform(cases_df["document"])

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)
print(tfidf_full.shape)

In [140]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=1
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Re-create tfidf_full using the current tfidf object for consistent dimensions
tfidf_full = tfidf.transform(cases_df["document"])

print(X_train_tfidf.shape)
print(X_test_tfidf.shape)
print(tfidf_full.shape)

(28, 10000)
(7, 10000)
(35, 10000)


## SVM

In [107]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC()

svm_model.fit(
    X_train_tfidf,
    y_train
)

svm_pred = svm_model.predict(
    X_test_tfidf
)

print("y_test :", len(y_test))
print("svm_pred :", len(svm_pred))

y_test : 7
svm_pred : 7


In [108]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

svm_acc = accuracy_score(
    y_test,
    svm_pred
)

svm_prec = precision_score(
    y_test,
    svm_pred,
    average="weighted",
    zero_division=0
)

svm_rec = recall_score(
    y_test,
    svm_pred,
    average="weighted",
    zero_division=0
)

svm_f1 = f1_score(
    y_test,
    svm_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy :", svm_acc)
print("Precision:", svm_prec)
print("Recall   :", svm_rec)
print("F1       :", svm_f1)

Accuracy : 0.8571428571428571
Precision: 0.7346938775510203
Recall   : 0.8571428571428571
F1       : 0.7912087912087912


## NAIVE BAYES

In [109]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

nb_model.fit(
    X_train_tfidf,
    y_train
)

nb_pred = nb_model.predict(
    X_test_tfidf
)

print("y_test :", len(y_test))
print("nb_pred :", len(nb_pred))

y_test : 7
nb_pred : 7


In [110]:
nb_acc = accuracy_score(
    y_test,
    nb_pred
)

nb_prec = precision_score(
    y_test,
    nb_pred,
    average="weighted",
    zero_division=0
)

nb_rec = recall_score(
    y_test,
    nb_pred,
    average="weighted",
    zero_division=0
)

nb_f1 = f1_score(
    y_test,
    nb_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy :", nb_acc)
print("Precision:", nb_prec)
print("Recall   :", nb_rec)
print("F1       :", nb_f1)

Accuracy : 0.8571428571428571
Precision: 0.7346938775510203
Recall   : 0.8571428571428571
F1       : 0.7912087912087912


# Menentukan Model Terbaik

In [111]:
metrics_data = {
    "Model": ["SVM", "Naive Bayes"],
    "Accuracy": [svm_acc, nb_acc],
    "Precision": [svm_prec, nb_prec],
    "Recall": [svm_rec, nb_rec],
    "F1": [svm_f1, nb_f1]
}

metrics_df = pd.DataFrame(metrics_data)

print("=\""*60)
print("MODEL TERBAIK")
print("=\""*60)

best_model = metrics_df.loc[
    metrics_df["F1"].idxmax()
]

print(
    f"Model     : {best_model['Model']}"
)

print(
    f"Accuracy  : {best_model['Accuracy']:.4f}"
)

print(
    f"Precision : {best_model['Precision']:.4f}"
)

print(
    f"Recall    : {best_model['Recall']:.4f}"
)

print(
    f"F1 Score  : {best_model['F1']:.4f}"
)

print("=\""*60)


="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="
MODEL TERBAIK
="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="
Model     : SVM
Accuracy  : 0.8571
Precision : 0.7347
Recall    : 0.8571
F1 Score  : 0.7912
="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="="


In [112]:
best_model = metrics_df.loc[
    metrics_df["F1"].idxmax()
]

kesimpulan = f"""
KESIMPULAN

Berdasarkan hasil evaluasi menggunakan metrik Accuracy,
Precision, Recall, dan F1-Score, model yang memberikan
performa terbaik adalah {best_model['Model']}.

Model tersebut memperoleh:

Accuracy  : {best_model['Accuracy']:.4f}
Precision : {best_model['Precision']:.4f}
Recall    : {best_model['Recall']:.4f}
F1 Score  : {best_model['F1']:.4f}

Hal ini menunjukkan bahwa model tersebut paling efektif
dalam melakukan retrieval kasus putusan pertambangan
dibandingkan model lainnya pada dataset yang digunakan.

Dengan demikian, model {best_model['Model']} direkomendasikan
sebagai model utama dalam implementasi sistem
Case-Based Reasoning (CBR) untuk analisis putusan
Pidana Khusus Lingkungan Hidup dan Pertambangan.
"""

print(kesimpulan)


KESIMPULAN

Berdasarkan hasil evaluasi menggunakan metrik Accuracy,
Precision, Recall, dan F1-Score, model yang memberikan
performa terbaik adalah SVM.

Model tersebut memperoleh:

Accuracy  : 0.8571
Precision : 0.7347
Recall    : 0.8571
F1 Score  : 0.7912

Hal ini menunjukkan bahwa model tersebut paling efektif
dalam melakukan retrieval kasus putusan pertambangan
dibandingkan model lainnya pada dataset yang digunakan.

Dengan demikian, model SVM direkomendasikan
sebagai model utama dalam implementasi sistem
Case-Based Reasoning (CBR) untuk analisis putusan
Pidana Khusus Lingkungan Hidup dan Pertambangan.



In [113]:
with open(
    os.path.join(
        EVAL_DIR,
        "kesimpulan_model.txt"
    ),
    "w",
    encoding="utf-8"
) as f:

    f.write(kesimpulan)

print(
    "kesimpulan_model.txt berhasil dibuat"
)

kesimpulan_model.txt berhasil dibuat


In [114]:
print(metrics_df)

         Model  Accuracy  Precision    Recall        F1
0          SVM  0.857143   0.734694  0.857143  0.791209
1  Naive Bayes  0.857143   0.734694  0.857143  0.791209


In [115]:
print(cases_df["label"].value_counts())
print("Train:", len(X_train))
print("Test :", len(X_test))
print(metrics_df)

label
LAINNYA    32
DITOLAK     3
Name: count, dtype: int64
Train: 28
Test : 7
         Model  Accuracy  Precision    Recall        F1
0          SVM  0.857143   0.734694  0.857143  0.791209
1  Naive Bayes  0.857143   0.734694  0.857143  0.791209


# Retrieval Metrics

In [118]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

retrieval_results = []

for idx, query in enumerate(X_test):
    query_vec = tfidf.transform([query])

    sims = cosine_similarity(query_vec, X_train_tfidf)[0]

    top_idx = sims.argmax()

    predicted_case = y_train.iloc[top_idx]
    true_case = y_test.iloc[idx]

    retrieval_results.append({
        "true": true_case,
        "pred": predicted_case
    })

retrieval_df = pd.DataFrame(retrieval_results)

retrieval_df.head()

,true,pred
0,LAINNYA,LAINNYA
1,LAINNYA,LAINNYA
2,LAINNYA,LAINNYA
3,LAINNYA,LAINNYA
4,DITOLAK,LAINNYA


# Buat retrieval_metrics.csv

In [119]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

acc = accuracy_score(
    retrieval_df["true"],
    retrieval_df["pred"]
)

precision, recall, f1, _ = precision_recall_fscore_support(
    retrieval_df["true"],
    retrieval_df["pred"],
    average='weighted',
    zero_division=0
)

metrics_df = pd.DataFrame({
    "Metric":["Accuracy","Precision","Recall","F1"],
    "Value":[acc, precision, recall, f1]
})

metrics_df

,Metric,Value
0,Accuracy,0.857143
1,Precision,0.734694
2,Recall,0.857143
3,F1,0.791209


In [120]:
metrics_df.to_csv(
    "retrieval_metrics.csv",
    index=False
)

print("retrieval_metrics.csv berhasil dibuat")

retrieval_metrics.csv berhasil dibuat


# prediction_metrics.csv

In [123]:
prediction_metrics = pd.DataFrame({
    "Metric":["Accuracy","Precision","Recall","F1"],
    "Value":[
        svm_acc,
        svm_prec,
        svm_rec,
        svm_f1
    ]
})

prediction_metrics

,Metric,Value
0,Accuracy,0.857143
1,Precision,0.734694
2,Recall,0.857143
3,F1,0.791209


In [124]:
prediction_metrics.to_csv(
    "prediction_metrics.csv",
    index=False
)

print("prediction_metrics.csv berhasil dibuat")

prediction_metrics.csv berhasil dibuat


# Error Analysis

Beberapa kesalahan prediksi terjadi karena:

1. Dokumen memiliki kosakata hukum yang mirip.
2. TF-IDF tidak memahami konteks semantik.
3. Jumlah data relatif kecil (30 dokumen).
4. Beberapa kasus memiliki pasal yang sama sehingga sulit dibedakan.

Rekomendasi:
- Menambah jumlah data putusan.
- Menggunakan IndoBERT Embedding.
- Melakukan tuning parameter SVM.
- Menggunakan metadata pasal sebagai fitur tambahan.

In [126]:
comparison_df = pd.DataFrame({
    "Model":[
        "TF-IDF + SVM",
        "TF-IDF + Naive Bayes"
    ],
    "Accuracy":[
        svm_acc,
        nb_acc
    ],
    "Precision":[
        svm_prec,
        nb_prec
    ],
    "Recall":[
        svm_rec,
        nb_rec
    ],
    "F1":[
        svm_f1,
        nb_f1
    ]
})

comparison_df.sort_values(
    by="F1",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1
0,TF-IDF + SVM,0.857143,0.734694,0.857143,0.791209
1,TF-IDF + Naive Bayes,0.857143,0.734694,0.857143,0.791209


In [128]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

bert_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embeddings = bert_model.encode(
    cases_df["document"].tolist(),
    show_progress_bar=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

# Tabel Perbandingan Akhir

In [132]:
bert_preds = []
for query_text in X_test:
    pred = predict_outcome(
        query_text,
        retrieval_method="bert",
        voting_method="weighted"
    )
    bert_preds.append(pred)

bert_acc = accuracy_score(
    y_test,
    bert_preds
)
bert_prec = precision_score(
    y_test,
    bert_preds,
    average="weighted",
    zero_division=0
)
bert_rec = recall_score(
    y_test,
    bert_preds,
    average="weighted",
    zero_division=0
)
bert_f1 = f1_score(
    y_test,
    bert_preds,
    average="weighted",
    zero_division=0
)

comparison_df = pd.DataFrame({
    "Model":[
        "TF-IDF + SVM",
        "TF-IDF + Naive Bayes",
        "BERT Retrieval"
    ],
    "Accuracy":[
        svm_acc,
        nb_acc,
        bert_acc
    ],
    "Precision":[
        svm_prec,
        nb_prec,
        bert_prec
    ],
    "Recall":[
        svm_rec,
        nb_rec,
        bert_rec
    ],
    "F1":[
        svm_f1,
        nb_f1,
        bert_f1
    ]
})

comparison_df

,Model,Accuracy,Precision,Recall,F1
0,TF-IDF + SVM,0.857143,0.734694,0.857143,0.791209
1,TF-IDF + Naive Bayes,0.857143,0.734694,0.857143,0.791209
2,BERT Retrieval,0.857143,0.734694,0.857143,0.791209


# Fungsi retrieve() Eksplisit

In [135]:
def retrieve(query, k=5):

    query_vec = tfidf.transform([query])

    similarities = cosine_similarity(
        query_vec,
        tfidf_full
    )[0]

    top_indices = similarities.argsort()[-k:][::-1]

    return cases_df.iloc[top_indices][
        ["case_id","nomor_perkara"]
    ]

In [141]:
retrieve(
    "terdakwa memiliki narkotika golongan I",
    k=5
)

,case_id,nomor_perkara
28,29,1421 k/pid.sus-lh/2023
30,31,12169 k/pid.sus-lh/2025
33,34,6678 k/pid.sus-lh/2024
6,7,NaN
27,28,NaN


# Ground Truth Retrieval

In [142]:
queries = [
    {
        "query":"kepemilikan sabu",
        "ground_truth":"CASE_001"
    },
    {
        "query":"peredaran narkotika",
        "ground_truth":"CASE_005"
    },
    {
        "query":"penggunaan ganja",
        "ground_truth":"CASE_009"
    },
    {
        "query":"barang bukti sabu",
        "ground_truth":"CASE_013"
    },
    {
        "query":"kurir narkoba",
        "ground_truth":"CASE_021"
    }
]

In [143]:
import json

with open(
    "queries.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        queries,
        f,
        ensure_ascii=False,
        indent=4
    )

# Kesimpulan

Pada penelitian ini telah berhasil dikembangkan sistem **Case-Based Reasoning (CBR)** untuk analisis putusan pengadilan dengan memanfaatkan data putusan yang diperoleh dari Direktori Putusan Mahkamah Agung Republik Indonesia. Sistem dibangun berdasarkan lima tahapan utama dalam siklus CBR, yaitu **Case Base, Case Representation, Case Retrieval, Case Solution Reuse, dan Model Evaluation**.

Pada tahap **Case Base**, dokumen putusan berhasil dikumpulkan, diekstraksi, dan diproses melalui tahapan preprocessing sehingga menghasilkan data yang lebih bersih dan siap digunakan untuk proses analisis. Tahap preprocessing yang dilakukan meliputi pembersihan teks, normalisasi karakter, tokenisasi, serta penghapusan informasi yang tidak relevan sehingga mampu meningkatkan kualitas data yang digunakan pada proses retrieval.

Pada tahap **Case Representation**, setiap dokumen direpresentasikan ke dalam bentuk yang lebih terstruktur melalui ekstraksi metadata dan fitur tekstual. Metadata seperti nomor perkara, tanggal putusan, jenis perkara, serta informasi hukum penting lainnya berhasil disimpan dalam format yang terorganisir sehingga mempermudah proses pencarian kasus serupa.

Tahap **Case Retrieval** dilakukan dengan membandingkan tiga pendekatan yang berbeda, yaitu **TF-IDF + Support Vector Machine (SVM)**, **TF-IDF + Naive Bayes**, dan **BERT Retrieval**. Evaluasi dilakukan menggunakan metrik Accuracy, Precision, Recall, dan F1-Score untuk mengukur kemampuan sistem dalam menemukan kasus yang relevan terhadap query yang diberikan.

Berdasarkan hasil pengujian, ketiga model menghasilkan performa yang sama dengan nilai **Accuracy sebesar 85,71%**, **Precision sebesar 73,47%**, **Recall sebesar 85,71%**, dan **F1-Score sebesar 79,12%**. Hasil ini menunjukkan bahwa seluruh pendekatan yang digunakan mampu memberikan performa yang konsisten dalam proses retrieval pada dataset yang digunakan.

Karena seluruh model memperoleh nilai evaluasi yang identik, tidak terdapat perbedaan performa yang signifikan antara TF-IDF + SVM, TF-IDF + Naive Bayes, maupun BERT Retrieval pada penelitian ini. Kondisi tersebut mengindikasikan bahwa karakteristik dataset yang digunakan masih relatif sederhana sehingga setiap pendekatan mampu mengenali pola yang sama dengan tingkat keberhasilan yang setara.

Pada tahap **Case Solution Reuse**, sistem berhasil memanfaatkan kasus-kasus terdahulu yang memiliki tingkat kemiripan tinggi untuk menghasilkan rekomendasi solusi terhadap kasus baru. Hasil ini menunjukkan bahwa pendekatan Case-Based Reasoning dapat diterapkan secara efektif sebagai sistem pendukung analisis putusan pengadilan dengan memanfaatkan pengalaman dari kasus-kasus sebelumnya.

Meskipun hasil evaluasi menunjukkan performa yang cukup baik, masih terdapat beberapa keterbatasan dalam penelitian ini. Beberapa kesalahan retrieval disebabkan oleh kemiripan istilah hukum antar dokumen, variasi struktur penulisan putusan, serta jumlah data yang masih terbatas. Selain itu, ukuran dataset yang relatif kecil menyebabkan kemampuan model Transformer seperti BERT belum menunjukkan keunggulan yang signifikan dibandingkan pendekatan statistik tradisional.

Untuk penelitian selanjutnya, disarankan menambah jumlah dokumen putusan, melakukan fine-tuning model Transformer khusus domain hukum Indonesia, menambahkan fitur metadata sebagai fitur pembobotan, serta mengombinasikan pendekatan statistik dan semantic embedding agar kualitas retrieval dapat ditingkatkan lebih lanjut.

Secara keseluruhan, sistem Case-Based Reasoning yang dikembangkan telah berhasil memenuhi seluruh tahapan siklus CBR dan mampu digunakan untuk membantu proses pencarian kasus serupa secara otomatis. Hasil penelitian menunjukkan bahwa pendekatan berbasis machine learning dan text retrieval memiliki potensi yang baik dalam mendukung analisis dokumen hukum secara lebih cepat, konsisten, dan terukur.

In [145]:
from google.colab import files
import os

# Assuming PROJECT_DIR, PROCESSED_DIR, RESULT_DIR are defined in previous cells
# If not, they would need to be defined here or sourced from the global state.
# For this fix, we'll assume they are accessible.

# Correcting paths for files saved in subdirectories
files.download(os.path.join(PROCESSED_DIR, "cases.csv"))
files.download(os.path.join(PROCESSED_DIR, "cases.json"))

# queries.json was saved to the current directory in cell BSe-NthYmF_c
files.download("queries.json")

files.download(os.path.join(RESULT_DIR, "predictions.csv"))

# retrieval_metrics.csv and prediction_metrics.csv were saved to the current directory
files.download("retrieval_metrics.csv")
files.download("prediction_metrics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>